# 🚀 Train Unified Gurukul Lite V2

**Train ONE adapter on 29 languages (21 original + 8 new)**

- **Expected Time:** 4-6 hours on T4 GPU (Colab free tier)
- **Output:** `gurukul_lite_v2` adapter supporting 38 languages total

---

## Setup Instructions

1. **Compress training data on your PC:**
   ```
   cd C:\pc\Project
   # Right-click data/training_merged/ → Send to → Compressed folder (.zip or .rar)
   # Supports: .zip, .rar, .tar.gz
   ```

2. **Upload to Colab:** Use Files panel (📁) to upload your compressed file

3. **Run all cells**

4. **Download** the trained adapter at the end

## 1️⃣ Check GPU

In [ ]:
!nvidia-smi

## 2️⃣ Check RAM & Install Dependencies

In [ ]:
import psutil

# Check available RAM
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"💾 Total RAM: {ram_gb:.1f} GB")

if ram_gb < 12:
    print("⚠️  WARNING: Less than 12GB RAM. Training may be slow.")
    print("   Consider using Colab Pro for more RAM.")
else:
    print("✅ Sufficient RAM for training")

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes sentencepiece psutil

## 3️⃣ Upload & Extract Training Data

Upload your compressed file using the Files panel (📁), then run the cell below.

**Supports:** `.zip`, `.rar`, `.tar.gz`, `.7z`

In [ ]:
import os

# Install unrar if needed
!apt-get install -qq unrar

# Auto-detect and extract compressed file
files = os.listdir('.')
compressed_file = None

for f in files:
    if f.startswith('training_merged'):
        compressed_file = f
        break

if compressed_file:
    print(f"📦 Found: {compressed_file}")
    
    if compressed_file.endswith('.zip'):
        !unzip -q {compressed_file} -d data/
    elif compressed_file.endswith('.rar'):
        !unrar x -y {compressed_file} data/
    elif compressed_file.endswith('.tar.gz') or compressed_file.endswith('.tgz'):
        !tar -xzf {compressed_file} -C data/
    elif compressed_file.endswith('.7z'):
        !7z x {compressed_file} -odata/
    
    print("\n✅ Extraction complete!")
    print("\n📁 Extracted files:")
    !ls -lh data/training_merged/ | head -n 35
else:
    print("❌ No training_merged file found!")
    print("Please upload: training_merged.zip, training_merged.rar, or training_merged.tar.gz")

## 4️⃣ Create Dataset (Memory-Efficient)

We'll use HuggingFace's `load_dataset` with streaming to avoid loading 10M+ samples into RAM.

In [ ]:
from pathlib import Path
from datasets import load_dataset

TRAIN_DATA_DIR = Path("data/training_merged")

print("📥 Scanning training data...")
train_files = sorted(TRAIN_DATA_DIR.glob("*.txt"))

print(f"\n📁 Found {len(train_files)} language files:")
for file_path in train_files:
    file_size_mb = file_path.stat().st_size / (1024 * 1024)
    print(f"   ✅ {file_path.stem:20} - {file_size_mb:6.1f} MB")

print("\n🔄 Creating dataset from text files (streaming mode)...")
train_dataset = load_dataset(
    'text',
    data_files={'train': [str(f) for f in train_files]},
    split='train',
    cache_dir='./cache'
)

print(f"\n✅ Dataset created with {len(train_dataset):,} examples")
print(f"📊 Total languages: {len(train_files)}")

## 5️⃣ Setup Tokenizer

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "bigscience/bloomz-560m"
MAX_LENGTH = 256

print(f"📥 Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded")

## 6️⃣ Tokenize Dataset (Batched)

This processes data in chunks to avoid RAM issues.

In [ ]:
import gc

print(f"🔤 Tokenizing {len(train_dataset):,} samples...")
print("⚠️  This will take 30-60 minutes for 10M+ samples")

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length'
    )

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=1000,  # Process 1000 samples at a time
    remove_columns=["text"],
    num_proc=2,  # Use 2 CPU cores
    desc="Tokenizing"
)

# Clear memory
del train_dataset
gc.collect()

print(f"✅ Tokenization complete")

# Split 90/10 for train/validation
print(f"\n📊 Splitting train/validation (90/10)...")
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split_dataset['train']
val_data = split_dataset['test']

# Clear more memory
del tokenized_dataset, split_dataset
gc.collect()

print(f"   Train: {len(train_data):,} samples")
print(f"   Val:   {len(val_data):,} samples")

## 7️⃣ Load Base Model (8-bit)

In [ ]:
from transformers import AutoModelForCausalLM
from peft import prepare_model_for_kbit_training
import torch

print(f"📥 Loading base model: {BASE_MODEL}")
print("   (Using 8-bit quantization to save GPU memory)")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    load_in_8bit=True,
    device_map="auto",
    torch_dtype=torch.float16
)

# Prepare model for 8-bit training with LoRA
print("🔧 Preparing model for 8-bit training...")
model = prepare_model_for_kbit_training(model)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

print(f"✅ Model loaded and prepared for training")

## 8️⃣ Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

print("⚙️  Configuring LoRA...")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query_key_value"],
    bias="none"
)

model = get_peft_model(model, lora_config)

print("✅ LoRA configured (r=8, alpha=16)")
print("\n📊 Trainable parameters:")
model.print_trainable_parameters()

## 9️⃣ Setup Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = "gurukul_lite_v2"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=200,
    logging_steps=100,
    save_steps=1000,
    eval_steps=1000,
    evaluation_strategy="steps",
    save_total_limit=3,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    report_to="none",
    gradient_checkpointing=True,
    optim="adamw_torch"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator
)

print("✅ Training configured")
print(f"   Epochs: 3")
print(f"   Batch size: 4 (effective: 16)")
print(f"   Total steps: ~{len(train_data) // 16 * 3:,}")
print(f"   Estimated time: 4-6 hours on T4")

## 🚀 START TRAINING

**This will take 4-6 hours.**

To prevent Colab disconnection:
1. Press F12 to open browser console
2. Paste and press Enter:
```javascript
function KeepClicking(){
  console.log("Clicking");
  document.querySelector("colab-connect-button").click()
}
setInterval(KeepClicking, 60000)
```

In [ ]:
from datetime import datetime

print("="*80)
print("🚀 STARTING TRAINING")
print("="*80)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Training on {len(train_data):,} samples across 29 languages")
print("="*80 + "\n")

start_time = datetime.now()

# Train!
trainer.train()

end_time = datetime.now()
duration = (end_time - start_time).total_seconds() / 3600

print("\n" + "="*80)
print("✅ TRAINING COMPLETE!")
print("="*80)
print(f"Duration: {duration:.2f} hours")
print(f"Finished: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

## 💾 Save Final Adapter

In [ ]:
import json

print("💾 Saving final adapter...")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save metadata
metadata = {
    "base_model": BASE_MODEL,
    "languages": 29,
    "total_languages_supported": 38,
    "train_samples": len(train_data),
    "val_samples": len(val_data),
    "epochs": 3,
    "lora_r": 8,
    "lora_alpha": 16,
    "training_duration_hours": round(duration, 2),
    "created": datetime.now().isoformat()
}

with open(f"{OUTPUT_DIR}/metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ Adapter saved to: {OUTPUT_DIR}/")
print("\n📋 Files created:")
!ls -lh {OUTPUT_DIR}/

## 🧪 Quick Test

Test the adapter before downloading:

In [ ]:
from peft import PeftModel

# Load fresh model + adapter
test_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16
)
test_model = PeftModel.from_pretrained(test_model, OUTPUT_DIR)

# Test prompts
test_prompts = [
    "भारत की राजधानी",  # Hindi
    "The capital of India is",  # English
    "ভারতের রাজধানী",  # Bengali
    "ආසියාවේ විශාලතම",  # Sinhala (new)
    "ประเทศไทยมีเมืองหลวง",  # Thai (new)
]

print("🧪 Testing adapter on multiple languages:\n")

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(test_model.device)
    outputs = test_model.generate(**inputs, max_new_tokens=30, temperature=0.7)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Output: {generated}\n")

print("✅ Test complete!")

## 📦 Download Adapter

In [ ]:
!zip -r gurukul_lite_v2.zip gurukul_lite_v2/
print("\n✅ Zipped! Download 'gurukul_lite_v2.zip' from Files panel (📁)")
!ls -lh gurukul_lite_v2.zip

---

## 🎉 Done!

**Your unified adapter is ready!**

### Next Steps:
1. Download `gurukul_lite_v2.zip`
2. Extract to `C:\pc\Project\adapters\gurukul_lite_v2\`
3. Test with your API server
4. Commit to repository

### What You Have:
- ✅ ONE adapter supporting **38 languages**
- ✅ Trained on **29 unique languages** (21 original + 8 new)
- ✅ Compatible with **9 bootstrapped dialects**
- ✅ Ready for production!